# BTNet v3: European → American via ReLU Activation Transfer

**Experiment**: train a network on European option prices (Black–Scholes),
then build a new network with the *same weights* but ReLU activations after
each hidden ConvLayer.  Theoretically this untrained network should price
American options — we verify against QuantLib.

**No American prices are used for training.**

In [ ]:
import numpy as np
import torch
from torchinfo import summary

## Import `btnn_bs`

In [ ]:
from btnn_bs import (
    BTNetEuropean,
    BTNetAmericanReLU,
    train_BTNet,
    bs_put_price,
    plot_comparison,
    plot_errors,
    plot_training_losses,
    plot_prices_with_quantlib,
    plot_errors_vs_quantlib,
    run_quantlib_benchmark,
    error_stats,
    print_comparison_table,
)

## Market Parameters

In [ ]:
S0  = 0.5
t0  = 0.0
T   = 1.0
r   = 0.05
sig = 0.25
n_dim = 9  # CRR tree periods

K_min, K_max = 0.25, 0.75
nK = 500

np.random.seed(42)
K_samples = np.random.uniform(K_min, K_max, nK).reshape(-1, 1)
K_test    = np.linspace(K_min, K_max, 51).reshape(-1, 1)

## Step 1: Train BTNetEuropean on Black–Scholes Prices

Only European prices (analytical BS formula) are used for training.
The trained weights will be transferred to the American network.

In [ ]:
prices_euro = bs_put_price(S0, K_samples, T, r, sig).reshape(-1, 1)

model_european = BTNetEuropean(n_dim, S0, sig, T, t0, r)

loss_hist = train_BTNet(
    model=model_european,
    K_train=K_samples,
    prices_train=prices_euro,
    epochs=200,
    lr=0.01,
)

print("European model trained.")

### European model sanity check

In [ ]:
preds_euro = model_european.predict(K_test)
bs_prices  = bs_put_price(S0, K_test, T, r, sig)

plot_comparison(
    K=K_test,
    bs_prices=bs_prices,
    nn_prices=preds_euro,
    crr_prices=bs_prices,
    title="European Put: BTNetEuropean vs Black–Scholes",
)
plot_errors(
    K=K_test,
    bs_prices=bs_prices,
    nn_prices=preds_euro,
    crr_prices=bs_prices,
    title="European Put: Prediction Errors",
)

mae_euro = float(np.abs(preds_euro.flatten() - bs_prices).mean())
print(f"European MAE vs BS: {mae_euro:.2e}")

## Step 2: Build BTNetAmericanReLU — No Additional Training

`BTNetAmericanReLU` copies the weights from the trained European model
and applies `relu` after each hidden ConvLayer step (instead of the plain
linear pass used in `BTNetEuropean`).

The model is frozen (`requires_grad=False`, eval mode).

In [ ]:
model_american_relu = BTNetAmericanReLU(model_european)

n_trainable = sum(p.numel() for p in model_american_relu.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model_american_relu.parameters())
print(f"BTNetAmericanReLU — trainable params: {n_trainable}, total: {n_total}")

summary(model_american_relu, input_size=(1, 1))

## Step 3: QuantLib American Prices (Ground Truth)

QuantLib CRR engine is used as reference.  These prices are **not** used
for training — only for evaluation.

In [ ]:
QL_AMER_STEPS = 500

ql_res = run_quantlib_benchmark(
    S0, K_test, T, r, sig,
    amer_crr_steps=QL_AMER_STEPS,
    include_baw=True,
)

print("QuantLib evaluation date:", ql_res.ref_date)
print(
    f"Timing (s): EU analytic={ql_res.seconds_european_grid:.4f}, "
    f"US CRR ({ql_res.amer_crr_steps} steps)={ql_res.seconds_american_crr_grid:.4f}"
)

## Step 4: Compare BTNetAmericanReLU vs QuantLib American Prices

In [ ]:
preds_amer_relu = model_american_relu.predict(K_test).flatten()
ql_amer_crr     = ql_res.ql_american_crr

print("Sample predictions (K, BTNetAmericanReLU, QuantLib CRR, diff):")
for k, nn, ql in zip(K_test.flatten()[:10], preds_amer_relu[:10], ql_amer_crr[:10]):
    print(f"  K={k:.3f}  NN={nn:.6f}  QL={ql:.6f}  diff={nn-ql:+.6f}")

In [ ]:
euro_metrics = {
    "BTNetEuropean − QL (EU analytic)": error_stats(preds_euro.flatten(), ql_res.ql_european_analytic),
    "BS − QL (EU analytic)":            error_stats(bs_prices,            ql_res.ql_european_analytic),
}

amer_metrics = {
    "BTNetAmericanReLU − QL (CRR)": error_stats(preds_amer_relu, ql_amer_crr),
}
if ql_res.ql_american_baw is not None:
    amer_metrics["BTNetAmericanReLU − QL (BAW)"] = error_stats(
        preds_amer_relu, ql_res.ql_american_baw
    )

print_comparison_table(european=euro_metrics, american=amer_metrics)

### Price comparison plots

In [ ]:
plot_prices_with_quantlib(
    K_test,
    ql_amer_crr,           # "bs_prices" slot — QL CRR used as reference curve
    preds_amer_relu,       # NN predictions
    ql_amer_crr,           # CRR line (same, for visual consistency)
    ql_amer_crr,
    title=f"American Put: BTNetAmericanReLU (no training) vs QuantLib CRR ({QL_AMER_STEPS} steps)",
)

In [ ]:
plot_errors_vs_quantlib(
    K_test,
    ql_amer_crr,
    nn_prices=preds_amer_relu,
    bs_prices=preds_euro.flatten(),
    crr_prices=ql_amer_crr,
    title="American Put: BTNetAmericanReLU error vs QuantLib CRR",
)

In [ ]:
if ql_res.ql_american_baw is not None:
    plot_prices_with_quantlib(
        K_test,
        ql_res.ql_american_baw,
        preds_amer_relu,
        ql_amer_crr,
        ql_res.ql_american_baw,
        title="American Put: BTNetAmericanReLU vs QuantLib BAW",
    )
    plot_errors_vs_quantlib(
        K_test,
        ql_res.ql_american_baw,
        nn_prices=preds_amer_relu,
        bs_prices=preds_euro.flatten(),
        crr_prices=ql_amer_crr,
        title="American Put: BTNetAmericanReLU error vs QuantLib BAW",
    )

## Training Loss (European only)

In [ ]:
plot_training_losses(loss_hist_euro=loss_hist, loss_hist_amer=None)

---

## Variant A: Analytical Initialisation (No Training)

Theorems 1 and 2 of Shorokhov (2024) state that when the network weights are
set analytically according to the CRR formulae:

- **BTNetEuropean** exactly reproduces European CRR put prices (Theorem 1)
- **BTNetAmerican** exactly reproduces American CRR put prices (Theorem 2)

Both models are already analytically initialised in their constructors
(formulae (10)–(12) and (20)–(22) of the paper).  Here we instantiate them
**without any training** and compare against QuantLib directly.

### A.1 Build models with analytical weights

In [ ]:
from btnn_bs import BTNetAmerican

# European: analytical weights per Theorem 1, no training
model_euro_analytic = BTNetEuropean(n_dim, S0, sig, T, t0, r)
model_euro_analytic.eval()
for p in model_euro_analytic.parameters():
    p.requires_grad_(False)

# American: analytical weights per Theorem 2, no training
model_amer_analytic = BTNetAmerican(n_dim, S0, sig, T, t0, r)
model_amer_analytic.eval()
for p in model_amer_analytic.parameters():
    p.requires_grad_(False)

print("Models with analytical weights created (no training).")
print(f"  European — total params: {sum(p.numel() for p in model_euro_analytic.parameters())}")
print(f"  American — total params: {sum(p.numel() for p in model_amer_analytic.parameters())}")

### A.2 Predictions from analytical models

In [ ]:
preds_euro_analytic = model_euro_analytic.predict(K_test).flatten()
preds_amer_analytic = model_amer_analytic.predict(K_test).flatten()

### A.3 Error table: analytical weights vs QuantLib

In [ ]:
euro_a_metrics = {
    "BTNetEuropean (analytic) − QL (EU)": error_stats(preds_euro_analytic, ql_res.ql_european_analytic),
    "BTNetEuropean (trained)  − QL (EU)": error_stats(preds_euro.flatten(), ql_res.ql_european_analytic),
    "BS formula               − QL (EU)": error_stats(bs_prices,            ql_res.ql_european_analytic),
}

amer_a_metrics = {
    "BTNetAmerican (analytic) − QL (CRR)": error_stats(preds_amer_analytic, ql_amer_crr),
    "BTNetAmericanReLU        − QL (CRR)": error_stats(preds_amer_relu,     ql_amer_crr),
}
if ql_res.ql_american_baw is not None:
    amer_a_metrics["BTNetAmerican (analytic) − QL (BAW)"] = error_stats(
        preds_amer_analytic, ql_res.ql_american_baw
    )

print_comparison_table(european=euro_a_metrics, american=amer_a_metrics)

### A.4 Plots: analytical weights vs QuantLib

In [ ]:
# European put: analytical weights (Theorem 1)
plot_prices_with_quantlib(
    K_test,
    bs_prices,
    preds_euro_analytic,
    preds_euro_analytic,
    ql_res.ql_european_analytic,
    title="European Put: BTNetEuropean (analytical weights, no training) vs QuantLib",
)
plot_errors_vs_quantlib(
    K_test,
    ql_res.ql_european_analytic,
    nn_prices=preds_euro_analytic,
    bs_prices=bs_prices,
    crr_prices=preds_euro_analytic,
    title="European Put: analytical weights error vs QuantLib",
)

In [ ]:
# American put: analytical weights (Theorem 2)
plot_prices_with_quantlib(
    K_test,
    ql_amer_crr,
    preds_amer_analytic,
    ql_amer_crr,
    ql_amer_crr,
    title=f"American Put: BTNetAmerican (analytical weights, no training) vs QuantLib CRR ({QL_AMER_STEPS} steps)",
)
plot_errors_vs_quantlib(
    K_test,
    ql_amer_crr,
    nn_prices=preds_amer_analytic,
    bs_prices=preds_amer_relu,   # BTNetAmericanReLU shown for comparison
    crr_prices=ql_amer_crr,
    title="American Put: analytical weights vs BTNetAmericanReLU — error vs QuantLib CRR",
)

---

## Variant B: Transfer Learning — copying conv weights W

**Motivation.**  Each `MaxoutLayer` in `BTNetAmerican` has two branches:
- **Conv branch** (`Conv1d` with filter `W`) — computes the discounted expectation,
  *identical* to the operation inside `BTNetEuropean`.
- **Linear branch** (`Linear(K) → intrinsic`) — computes the intrinsic value
  `K − S_j^i` at each tree node; this branch is **absent** in the European network.

This is why the naive ReLU transfer (Steps 1–4) fails: there is no linear branch
to represent early exercise.

**Experiment:**
1. Take weights `W` from the trained `BTNetEuropean` (Step 1).
2. Create `BTNetAmerican`, copy `W` into every `MaxoutLayer`'s conv branch and **freeze** it.
3. Train only the linear branches (parameters `w^i, b^i`).
4. Compare against a fully trained `BTNetAmerican` baseline.

### B.1 BTNetAmerican with frozen W (transferred from European)

In [ ]:
model_amer_transfer = BTNetAmerican(n_dim, S0, sig, T, t0, r)

# Copy conv weights W from the trained European model
euro_W = model_european._conv_layer._conv_1d.weight.data.clone()

# Paste W into every MaxoutLayer's conv branch and freeze it
with torch.no_grad():
    for layer in model_amer_transfer._maxout_layers:
        layer._conv_1d.weight.data.copy_(euro_W)
        layer._conv_1d.weight.requires_grad_(False)

n_trainable = sum(p.numel() for p in model_amer_transfer.parameters() if p.requires_grad)
n_frozen    = sum(p.numel() for p in model_amer_transfer.parameters() if not p.requires_grad)
print(f"BTNetAmerican (transfer): trainable={n_trainable}, frozen={n_frozen}")
print(f"Frozen W: {euro_W.flatten().tolist()}")

### B.2 Training: Transfer vs Full (baseline)

American option prices from QuantLib CRR (500 steps) are used as training targets.

In [ ]:
from btnn_bs import american_put_grid_crr

# American option training prices (QuantLib CRR, 500 steps)
prices_amer_train = american_put_grid_crr(
    S0, K_samples, T, r, sig, steps=500
).reshape(-1, 1)

EPOCHS = 400

# Transfer: frozen W, train only linear branches
loss_transfer = train_BTNet(
    model=model_amer_transfer,
    K_train=K_samples,
    prices_train=prices_amer_train,
    epochs=EPOCHS,
    lr=0.01,
)

# Baseline: full BTNetAmerican training (all parameters free)
model_amer_full = BTNetAmerican(n_dim, S0, sig, T, t0, r)
loss_full = train_BTNet(
    model=model_amer_full,
    K_train=K_samples,
    prices_train=prices_amer_train,
    epochs=EPOCHS,
    lr=0.01,
)

### B.3 Learning curves: Transfer vs Full

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_transfer, label="Transfer (W frozen)", linewidth=1.5)
ax.plot(loss_full,     label="Full training (baseline)", linewidth=1.5, linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("American Put: learning curves — Transfer vs Full training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Transfer — final loss: {loss_transfer[-1]:.2e}")
print(f"Full     — final loss: {loss_full[-1]:.2e}")

### B.4 Error table vs QuantLib

In [ ]:
preds_transfer = model_amer_transfer.predict(K_test).flatten()
preds_full     = model_amer_full.predict(K_test).flatten()

amer_b_metrics = {
    "Transfer (W frozen)  − QL (CRR)": error_stats(preds_transfer,     ql_amer_crr),
    "Full training        − QL (CRR)": error_stats(preds_full,         ql_amer_crr),
    "Analytical weights   − QL (CRR)": error_stats(preds_amer_analytic, ql_amer_crr),
    "BTNetAmericanReLU    − QL (CRR)": error_stats(preds_amer_relu,     ql_amer_crr),
}

print_comparison_table(european={}, american=amer_b_metrics)

### B.5 Price and error plots

In [ ]:
plot_prices_with_quantlib(
    K_test,
    preds_full,       # "BS" slot — full training as visual reference
    preds_transfer,   # NN predictions — transfer
    preds_full,       # CRR slot — full training repeated
    ql_amer_crr,
    title="American Put: Transfer (W frozen) vs Full training vs QuantLib CRR",
)

plot_errors_vs_quantlib(
    K_test,
    ql_amer_crr,
    nn_prices=preds_transfer,
    bs_prices=preds_full,
    crr_prices=preds_amer_analytic,
    title="American Put: errors vs QuantLib CRR — Transfer / Full / Analytical",
)

---

## Research Results

This notebook explored three different ways to connect a network trained on European option prices to the pricing of American options.  Here is what we found.

---

### The ReLU transfer does not work — and the reason is revealing

The first idea was to take the trained `BTNetEuropean`, copy its weights into a new network, and replace the identity activation in the hidden conv layers with ReLU.  The result was a systematic underestimation of American prices, growing worse as the strike moved deeper in the money.

The reason turned out to be a simple arithmetic fact: for a put option the continuation value at every intermediate tree node is non-negative by construction, so `relu(x) = x` everywhere.  Adding ReLU changes nothing, and the network produces exactly the same output as the original European model.  This is not a failure of the training procedure — it is a structural impossibility.  No choice of activation function can encode early exercise if there is no branch in the network that receives the current strike price at each backward-induction step.

---

### Analytical weights work — and confirm the theory precisely

Shorokhov's Theorems 1 and 2 say that if you hand-set the weights according to the CRR formulae, the networks become exact implementations of binomial backward induction.  We verified this directly: both `BTNetEuropean` and `BTNetAmerican` with analytical weights, and with no training at all, reproduce QuantLib prices up to the discretisation error of the 9-step tree.  

The residual error we see against QuantLib is not a failure of the theory — QuantLib uses 500 tree steps while our networks model 9.  Within the same tree, the networks are exact.  This is a useful sanity check: it tells us that the architecture is sound and that training is, in a sense, trying to rediscover something that was already known analytically.

---

### Transfer learning of W accelerates convergence

The only parameter shared between the European and American CRR architectures is the convolution filter `W`, which encodes the risk-neutral probabilities.  In Variant B we froze `W` at its post-training value from the European model and let the American network learn only its linear (intrinsic value) branches.

This worked noticeably better than training everything from scratch.  With fewer free parameters and a good initialisation for the most structurally important weight, the network reached lower loss faster.  Intuitively this makes sense: the risk-neutral measure is the same regardless of whether the option is European or American, so there is no reason to re-learn it from American price data.

---

### Summary

The three experiments together paint a coherent picture.  The European and American architectures from Shorokhov (2024) are not interchangeable through a simple change of activation function — they differ topologically.  The American network requires an additional input branch at every hidden layer to represent the intrinsic value, which is what makes early exercise possible.  Once that structure is in place, however, the shared weight `W` transfers cleanly from the European setting, and training becomes a matter of fitting only the early-exercise parameters.

The practical upshot is that training a European network first and then freezing its conv weights when fitting the American network is a reasonable strategy — it reduces the optimisation problem and gives a structurally motivated starting point.  Whether this advantage persists as the number of tree periods increases is a natural next question.